In [ ]:
!pip install xgboost scikit-learn gradio matplotlib pandas numpy --quiet

In [ ]:
# SECTION 1 - IMPORTS
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import gradio as gr
import warnings
warnings.filterwarnings("ignore")

print("Imports complete")

# SECTION 2 - LOAD DATA
BASE = "/content"

restaurants = pd.read_csv(f"/content/restaurants.csv")
hourly = pd.read_csv(f"/content/hourly_demand.csv", encoding="ISO-8859-1")
staffing = pd.read_csv(f"/content/staffing_plan.csv", encoding="ISO-8859-1")
inventory = pd.read_csv(f"/content/inventory_plan.csv")
corrections = pd.read_csv(f"/content/manager_corrections.csv")

for df_ in [hourly, staffing, inventory, corrections]:
    df_["date"] = pd.to_datetime(df_["date"], errors="coerce")

for df_ in [hourly, staffing, inventory, corrections, restaurants]:
    if "restaurant_id" in df_.columns:
        df_["restaurant_id"] = df_["restaurant_id"].astype(str)

print(
    f"Data loaded - hourly:{hourly.shape} staff:{staffing.shape} "
    f"inv:{inventory.shape} corrections:{corrections.shape}"
)


def most_common_value(series, fallback):
    clean = series.dropna()
    if clean.empty:
        return fallback
    return clean.mode().iloc[0]

# SECTION 3 - DATA-DRIVEN CONSTANTS
_wbase = hourly.groupby("weather_type")["actual_covers"].mean()
DEFAULT_WEATHER = most_common_value(hourly["weather_type"], "Unknown")
WEATHER_MULT = (_wbase / _wbase.get(DEFAULT_WEATHER, _wbase.mean())).to_dict()

_etemp = hourly.copy()
DEFAULT_EVENT = most_common_value(_etemp["event_type"], "Missing")
_etemp["event_type"] = _etemp["event_type"].fillna(DEFAULT_EVENT)
_ebase = _etemp.groupby("event_type")["actual_covers"].mean()
EVENT_MULT = (_ebase / _ebase.get(DEFAULT_EVENT, _ebase.mean())).to_dict()

_ptemp = hourly.copy()
DEFAULT_PROMO = most_common_value(_ptemp["promo_type"], "Missing")
_ptemp["promo_type"] = _ptemp["promo_type"].fillna(DEFAULT_PROMO)
_pbase = _ptemp.groupby("promo_type")["actual_covers"].mean()
PROMO_MULT = (_pbase / _pbase.get(DEFAULT_PROMO, _pbase.mean())).to_dict()

HOUR_AVG = hourly.groupby(["restaurant_id", "hour"])["actual_covers"].mean().to_dict()
OPERATING_HOURS = sorted(int(h) for h in hourly["hour"].dropna().unique())

_hour_profile = hourly.groupby("hour")["actual_covers"].mean()
_daytime_candidates = [h for h in OPERATING_HOURS if 10 <= h <= 16]
_evening_candidates = [h for h in OPERATING_HOURS if 17 <= h <= 23]
LUNCH_PEAK_HOURS = sorted(
    int(h) for h in _hour_profile.reindex(_daytime_candidates).dropna().nlargest(3).index
)
DINNER_PEAK_HOURS = sorted(
    int(h) for h in _hour_profile.reindex(_evening_candidates).dropna().nlargest(3).index
)
if not LUNCH_PEAK_HOURS:
    LUNCH_PEAK_HOURS = OPERATING_HOURS[: max(1, min(3, len(OPERATING_HOURS)))]
if not DINNER_PEAK_HOURS:
    DINNER_PEAK_HOURS = OPERATING_HOURS[-max(1, min(3, len(OPERATING_HOURS))):]
ACTIVE_SERVICE_HOURS = sorted(set(LUNCH_PEAK_HOURS + DINNER_PEAK_HOURS))

DEFAULT_RESTAURANT = sorted(restaurants["restaurant_id"].astype(str).unique())[0]
DEFAULT_FORECAST_DATE = (
    hourly["date"].dropna().max() + pd.Timedelta(days=1)
).strftime("%Y-%m-%d")

WEATHER_STATS = (
    hourly.groupby("weather_type")[["rain_mm", "temperature_c"]]
    .mean()
    .to_dict(orient="index")
)
DEFAULT_RAIN_MM = float(hourly["rain_mm"].mean()) if "rain_mm" in hourly else 0.0
DEFAULT_TEMP_C = float(hourly["temperature_c"].mean()) if "temperature_c" in hourly else 25.0

_wkbase = hourly.groupby("is_weekend")["actual_covers"].mean()
WEEKEND_MULT = (_wkbase / _wkbase.get(0, _wkbase.mean())).to_dict()

print("\nData-driven constants:")
print(f"  Weather multipliers : {WEATHER_MULT}")
print(f"  Event multipliers   : {EVENT_MULT}")
print(f"  Promo multipliers   : {PROMO_MULT}")
print(f"  Weekend multiplier  : {WEEKEND_MULT}")
print(f"  Operating hours     : {OPERATING_HOURS}")
print(f"  Lunch peak hours    : {LUNCH_PEAK_HOURS}")
print(f"  Dinner peak hours   : {DINNER_PEAK_HOURS}")

# SECTION 4 - FEATURE ENGINEERING
print("\n" + "=" * 60)
print("  FEATURE ENGINEERING")
print("=" * 60)

le_weather = LabelEncoder()
le_event = LabelEncoder()
le_promo = LabelEncoder()
le_rest = LabelEncoder()

df = hourly.copy()
df["event_type"] = df["event_type"].fillna(DEFAULT_EVENT)
df["promo_type"] = df["promo_type"].fillna(DEFAULT_PROMO)
df["weather_type"] = df["weather_type"].fillna(DEFAULT_WEATHER)
df["restaurant_id"] = df["restaurant_id"].astype(str)

df["weather_enc"] = le_weather.fit_transform(df["weather_type"])
df["event_enc"] = le_event.fit_transform(df["event_type"])
df["promo_enc"] = le_promo.fit_transform(df["promo_type"])
df["rest_enc"] = le_rest.fit_transform(df["restaurant_id"])

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["is_lunch_peak"] = df["hour"].isin(LUNCH_PEAK_HOURS).astype(int)
df["is_dinner_peak"] = df["hour"].isin(DINNER_PEAK_HOURS).astype(int)
df["is_off_peak"] = (~df["hour"].isin(ACTIVE_SERVICE_HOURS)).astype(int)

df["weather_mult"] = df["weather_type"].map(WEATHER_MULT).fillna(1.0)
df["event_mult"] = df["event_type"].map(EVENT_MULT).fillna(1.0)
df["promo_mult"] = df["promo_type"].map(PROMO_MULT).fillna(1.0)
df["is_weekend_mult"] = df["is_weekend"].map(WEEKEND_MULT).fillna(1.0)

df["month"] = df["date"].dt.month
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["day_of_month"] = df["date"].dt.day
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

restaurants["restaurant_id"] = restaurants["restaurant_id"].astype(str)

df = df.merge(
    restaurants[["restaurant_id", "seating_capacity", "avg_ticket_size", "cuisine_type"]],
    on="restaurant_id",
    how="left",
)

df["cuisine_type"] = df["cuisine_type"].fillna("Unknown")
le_cuisine = LabelEncoder()
df["cuisine_enc"] = le_cuisine.fit_transform(df["cuisine_type"])
df["capacity_util"] = df["actual_covers"] / df["seating_capacity"].replace(0, np.nan)

df["hour_avg_rest"] = df.apply(
    lambda r: HOUR_AVG.get((r["restaurant_id"], r["hour"]), r["actual_covers"]),
    axis=1,
)

df = df.sort_values(["restaurant_id", "date", "hour"]).reset_index(drop=True)

df["lag_1h"] = df.groupby("restaurant_id")["actual_covers"].shift(1)
df["lag_24h"] = df.groupby("restaurant_id")["actual_covers"].shift(24)
df["lag_48h"] = df.groupby("restaurant_id")["actual_covers"].shift(48)
df["lag_168h"] = df.groupby("restaurant_id")["actual_covers"].shift(168)

df["roll_3h"] = df.groupby("restaurant_id")["actual_covers"].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)
df["roll_6h"] = df.groupby("restaurant_id")["actual_covers"].transform(
    lambda x: x.shift(1).rolling(6, min_periods=1).mean()
)
df["roll_24h"] = df.groupby("restaurant_id")["actual_covers"].transform(
    lambda x: x.shift(1).rolling(24, min_periods=1).mean()
)

corr_feat = corrections.copy()
corr_feat["restaurant_id"] = corr_feat["restaurant_id"].astype(str)
DEFAULT_REASON = most_common_value(corr_feat["correction_reason"], "Unspecified")
corr_feat["correction_reason"] = corr_feat["correction_reason"].fillna(DEFAULT_REASON)
corr_feat["correction_ratio"] = (
    corr_feat["actual_covers"] / corr_feat["predicted_covers"].replace(0, np.nan)
).fillna(1.0).clip(0.5, 2.0)

le_reason = LabelEncoder()
corr_feat["reason_enc"] = le_reason.fit_transform(corr_feat["correction_reason"])

df = df.merge(
    corr_feat[["restaurant_id", "date", "correction_ratio", "reason_enc"]],
    on=["restaurant_id", "date"],
    how="left",
)

df["correction_ratio"] = df["correction_ratio"].fillna(1.0)
df["reason_enc"] = df["reason_enc"].fillna(0).astype(int)

df["peak_weather"] = (df["is_lunch_peak"] + df["is_dinner_peak"]) * df["weather_mult"]
df["peak_event"] = (df["is_lunch_peak"] + df["is_dinner_peak"]) * df["event_mult"]

df = df.bfill().ffill().fillna(0)

FEATURES = [
    "rest_enc", "seating_capacity", "avg_ticket_size", "cuisine_enc",
    "hour", "hour_sin", "hour_cos",
    "day_of_week", "month_sin", "month_cos",
    "week_of_year", "day_of_month",
    "is_lunch_peak", "is_dinner_peak", "is_off_peak",
    "is_weekend", "is_weekend_mult",
    "weather_enc", "weather_mult",
    "event_enc", "event_mult",
    "promo_enc", "promo_mult",
    "rain_mm", "temperature_c",
    "peak_weather", "peak_event",
    "hour_avg_rest",
    "lag_1h", "lag_24h", "lag_48h", "lag_168h",
    "roll_3h", "roll_6h", "roll_24h",
    "correction_ratio", "reason_enc",
]
TARGET = "actual_covers"

print(f"Features engineered: {len(FEATURES)}")

# SECTION 5 - DEMAND MODEL TRAINING
print("\n" + "=" * 60)
print("  DEMAND FORECASTING MODEL - XGBoost v2")
print("=" * 60)

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.75,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.5,
    early_stopping_rounds=30,
    eval_metric="mae",
    random_state=42,
)

model.fit(
    X_train_sc,
    y_train,
    eval_set=[(X_test_sc, y_test)],
    verbose=False,
)

train_mae = round(mean_absolute_error(y_train, model.predict(X_train_sc).clip(0)), 2)
test_mae = round(mean_absolute_error(y_test, model.predict(X_test_sc).clip(0)), 2)
test_rmse = round(np.sqrt(mean_squared_error(y_test, model.predict(X_test_sc).clip(0))), 2)

print(f"  Best iteration : {model.best_iteration}")
print(f"  Train MAE      : {train_mae}")
print(f"  Test  MAE      : {test_mae} - held-out")
print(f"  Test  RMSE     : {test_rmse}")

fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\n  Top 10 features:")
for feat, imp in fi.head(10).items():
    print(f"    {feat:25s}  {imp:.4f}")

# SECTION 6 - STAFF PLANNING MODEL
print("\n" + "=" * 60)
print("  STAFF PLANNING MODEL - role x station")
print("=" * 60)

staff_merged = (
    staffing
    .merge(
        hourly[["restaurant_id", "date", "hour", "actual_covers", "day_of_week", "is_weekend"]],
        on=["restaurant_id", "date", "hour"],
        how="left",
    )
    .merge(
        restaurants[["restaurant_id", "seating_capacity"]],
        on="restaurant_id",
        how="left",
    )
    .fillna(0)
)

staff_merged["is_lunch_peak"] = staff_merged["hour"].isin(LUNCH_PEAK_HOURS).astype(int)
staff_merged["is_dinner_peak"] = staff_merged["hour"].isin(DINNER_PEAK_HOURS).astype(int)
staff_merged["hour_sin"] = np.sin(2 * np.pi * staff_merged["hour"] / 24)
staff_merged["hour_cos"] = np.cos(2 * np.pi * staff_merged["hour"] / 24)

STAFF_FEATURES = [
    "hour", "hour_sin", "hour_cos",
    "day_of_week", "is_weekend",
    "is_lunch_peak", "is_dinner_peak",
    "actual_covers", "seating_capacity",
]

staff_models = {}
staff_scalers = {}

for (role, station), grp in staff_merged.groupby(["role", "station"]):
    X = grp[STAFF_FEATURES]
    y = grp["staff_required"]

    sc = StandardScaler()
    m = XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42,
    )

    m.fit(sc.fit_transform(X), y)
    staff_models[(role, station)] = m
    staff_scalers[(role, station)] = sc

print(f"  Trained {len(staff_models)} role x station models")
print(f"  Roles   : {sorted(set(r for r, s in staff_models))}")
print(f"  Stations: {sorted(set(s for r, s in staff_models))}")

# SECTION 7 - INVENTORY MODEL
print("\n" + "=" * 60)
print("  INVENTORY MODEL - shelf-life & lead-time aware")
print("=" * 60)

inv_merged = (
    inventory
    .merge(
        hourly.groupby(["restaurant_id", "date"])["actual_covers"]
        .sum()
        .reset_index()
        .rename(columns={"actual_covers": "day_covers"}),
        on=["restaurant_id", "date"],
        how="left",
    )
    .merge(
        hourly.groupby(["restaurant_id", "date"])["is_weekend"]
        .first()
        .reset_index(),
        on=["restaurant_id", "date"],
        how="left",
    )
    .fillna(0)
)

inv_merged["safety_factor"] = np.where(
    inv_merged["shelf_life_days"] <= 3,
    0.10,
    np.where(inv_merged["shelf_life_days"] <= 7, 0.20, 0.30),
)
inv_merged["usable_days"] = np.minimum(inv_merged["shelf_life_days"], 7)
inv_merged["lead_urgency"] = (
    inv_merged["supplier_lead_time_days"] /
    inv_merged["shelf_life_days"].clip(lower=1)
).clip(0, 5)

INV_FEATURES = [
    "day_covers", "shelf_life_days", "supplier_lead_time_days",
    "is_weekend", "safety_factor", "usable_days", "lead_urgency",
    "projected_usage_qty",
]

ing_models = {}
ing_scalers = {}

for ing, grp in inv_merged.groupby("ingredient_name"):
    X = grp[INV_FEATURES]
    y = grp["order_qty"]

    sc = StandardScaler()
    m = XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42,
    )

    m.fit(sc.fit_transform(X), y)
    mae = round(mean_absolute_error(y, m.predict(sc.transform(X))), 3)

    ing_models[ing] = m
    ing_scalers[ing] = sc

    print(f"  {ing:12s}  MAE: {mae}")

# SECTION 8 - SELF-LEARNING FEEDBACK LOOP
print("\n" + "=" * 60)
print("  SELF-LEARNING FEEDBACK LOOP v2 - CORRECTED")
print("=" * 60)

convergence_log = []
live_feedback_log = []
feedback_memory = {}


def safe_encode(le, val, default=0):
    try:
        return int(le.transform([val])[0])
    except Exception:
        return default


def feedback_key(restaurant_id, weather, event, promo, is_weekend_flag):
    return (
        str(restaurant_id),
        str(weather),
        str(event),
        str(promo),
        int(is_weekend_flag),
    )


def get_calibration_factor(restaurant_id, weather, event, promo, is_weekend_flag):
    key = feedback_key(restaurant_id, weather, event, promo, is_weekend_flag)
    ratios = feedback_memory.get(key, [])

    if not ratios:
        rest_ratios = []
        for old_key, old_ratios in feedback_memory.items():
            if old_key[0] == str(restaurant_id):
                rest_ratios.extend(old_ratios[-3:])
        ratios = rest_ratios

    if not ratios:
        return 1.0

    recent = np.asarray(ratios[-6:], dtype=float)
    weights = np.linspace(1.0, 2.0, len(recent))
    factor = float(np.average(recent, weights=weights))
    return float(np.clip(factor, 0.45, 1.65))


def remember_feedback(restaurant_id, weather, event, promo, is_weekend_flag, ratio):
    key = feedback_key(restaurant_id, weather, event, promo, is_weekend_flag)
    feedback_memory.setdefault(key, []).append(float(np.clip(ratio, 0.45, 1.65)))
    feedback_memory[key] = feedback_memory[key][-12:]


def apply_feedback(
    restaurant_id: str,
    actual_total: float,
    pred_total: float,
    correction_reason: str = DEFAULT_REASON,
    is_live: bool = False,
    weather: str = DEFAULT_WEATHER,
    event: str = DEFAULT_EVENT,
    promo: str = DEFAULT_PROMO,
    is_weekend_flag: int = 0,
    forecast_rows: pd.DataFrame = None,
    forecast_covers=None,
) -> str:
    global df, model, scaler, convergence_log, live_feedback_log, feedback_memory

    if pred_total == 0:
        return "Skipped - no prediction available."

    ratio = actual_total / pred_total

    if ratio < 0.50 or ratio > 2.0:
        return (
            f"Rejected - ratio {ratio:.2f} outside safe range (0.50-2.0). "
            f"Predicted: {int(pred_total)}, Actual: {int(actual_total)}. "
            f"Enter a value between {int(pred_total * 0.5)} and {int(pred_total * 2.0)}"
        )

    df["restaurant_id"] = df["restaurant_id"].astype(str)
    rest_id_str = str(restaurant_id)

    remember_feedback(rest_id_str, weather, event, promo, is_weekend_flag, ratio)

    if forecast_rows is not None and len(forecast_rows) > 0:
        rest_rows = forecast_rows.copy()
        rest_rows["restaurant_id"] = rest_id_str
        if "date" not in rest_rows.columns:
            rest_rows["date"] = df["date"].max() + pd.Timedelta(days=1)
        rest_rows["actual_covers"] = 0.0

        if forecast_covers is not None and np.sum(forecast_covers) > 0:
            shape = np.asarray(forecast_covers, dtype=float)
            shape = shape / shape.sum()
        else:
            shape = np.ones(len(rest_rows), dtype=float) / len(rest_rows)

        rest_rows["actual_covers"] = actual_total * shape
        rest_rows["correction_ratio"] = np.clip(ratio, 0.5, 2.0)
        rest_rows["reason_enc"] = safe_encode(le_reason, correction_reason)
    else:
        rest_rows = df[df["restaurant_id"] == rest_id_str].tail(15).copy()

        if rest_rows.empty:
            rest_rows = df.tail(15).copy()
            rest_rows["restaurant_id"] = rest_id_str
            rest_rows["rest_enc"] = safe_encode(le_rest, rest_id_str)

        error_magnitude = abs(1.0 - ratio)
        alpha = min(0.95, 0.65 + error_magnitude * 0.7)

        n_hours = len(rest_rows)
        avg_actual = actual_total / max(1, n_hours)

        rest_rows["actual_covers"] = (
            (1 - alpha) * rest_rows["actual_covers"] + alpha * avg_actual
        ).clip(lower=0)

        rest_rows["correction_ratio"] = np.clip(ratio, 0.5, 2.0)

        try:
            rest_rows["reason_enc"] = int(le_reason.transform([correction_reason])[0])
        except Exception:
            rest_rows["reason_enc"] = 0

    error_magnitude = abs(1.0 - ratio)
    alpha = min(0.95, 0.65 + error_magnitude * 0.7)

    combined = pd.concat([df, rest_rows], ignore_index=True).tail(20000)
    combined["restaurant_id"] = combined["restaurant_id"].astype(str)
    combined = combined.sort_values(["restaurant_id", "date", "hour"]).reset_index(drop=True)

    combined["lag_1h"] = combined.groupby("restaurant_id")["actual_covers"].shift(1)
    combined["lag_24h"] = combined.groupby("restaurant_id")["actual_covers"].shift(24)
    combined["lag_48h"] = combined.groupby("restaurant_id")["actual_covers"].shift(48)
    combined["lag_168h"] = combined.groupby("restaurant_id")["actual_covers"].shift(168)

    combined["roll_3h"] = combined.groupby("restaurant_id")["actual_covers"].transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
    combined["roll_6h"] = combined.groupby("restaurant_id")["actual_covers"].transform(
        lambda x: x.shift(1).rolling(6, min_periods=1).mean()
    )
    combined["roll_24h"] = combined.groupby("restaurant_id")["actual_covers"].transform(
        lambda x: x.shift(1).rolling(24, min_periods=1).mean()
    )

    combined = combined.bfill().ffill().fillna(0)
    df = combined

    new_scaler = StandardScaler()
    X_new = df[FEATURES]
    y_new = df[TARGET]
    X_sc = new_scaler.fit_transform(X_new)

    sample_weights = np.ones(len(df))
    sample_weights[-len(rest_rows):] = 150.0

    new_model = XGBRegressor(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.75,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=1.5,
        random_state=42,
    )

    new_model.fit(
        X_sc,
        y_new,
        sample_weight=sample_weights,
        verbose=False,
    )

    scaler = new_scaler
    model = new_model

    global_mae = round(
        mean_absolute_error(y_new, model.predict(scaler.transform(X_new)).clip(0)),
        2,
    )

    total_error = round(abs(pred_total - actual_total), 2)
    total_error_pct = round((total_error / actual_total) * 100, 2) if actual_total else 0.0
    calibration_factor = get_calibration_factor(
        rest_id_str, weather, event, promo, is_weekend_flag
    )
    next_estimate = int(round(pred_total * calibration_factor))

    entry = {
        "restaurant": rest_id_str,
        "predicted": int(pred_total),
        "actual": int(actual_total),
        "ratio": round(ratio, 3),
        "alpha": round(alpha, 3),
        "global_mae": global_mae,
        "total_error": total_error,
        "total_error_pct": total_error_pct,
        "calibration_factor": round(calibration_factor, 3),
        "next_estimate": next_estimate,
        "reason": correction_reason,
    }

    if is_live:
        live_feedback_log.append({**entry, "round": len(live_feedback_log) + 1})
        round_label = f"Live Round: {len(live_feedback_log)}"
    else:
        convergence_log.append({**entry, "round": len(convergence_log) + 1})
        round_label = f"Replay Round: {len(convergence_log)}"

    return (
        f"Model retrained | {round_label} | "
        f"Ratio: {ratio:.2f} | Alpha: {alpha:.2f} | "
        f"Global MAE: {global_mae} | "
        f"Total Error: {total_error} covers ({total_error_pct}%) | "
        f"Next calibrated estimate: {next_estimate} | "
        f"Reason: {correction_reason}"
    )


# SECTION 9 - FORECAST ENGINE
def get_rest_meta(restaurant_id):
    row = restaurants[restaurants["restaurant_id"] == str(restaurant_id)].iloc[0]
    return row.to_dict()


def build_forecast_rows(restaurant_id, weather, event, promo, forecast_date, is_weekend_flag=None):
    meta = get_rest_meta(restaurant_id)
    forecast_ts = pd.to_datetime(forecast_date, errors="coerce")
    if pd.isna(forecast_ts):
        forecast_ts = hourly["date"].dropna().max() + pd.Timedelta(days=1)
    forecast_ts = pd.Timestamp(forecast_ts).normalize()

    if is_weekend_flag is None:
        is_weekend_flag = int(forecast_ts.dayofweek >= 5)
    else:
        is_weekend_flag = int(is_weekend_flag)

    day_of_week = int(forecast_ts.dayofweek)
    month = int(forecast_ts.month)
    week_of_year = int(forecast_ts.isocalendar().week)
    day_of_month = int(forecast_ts.day)

    w_enc = safe_encode(le_weather, weather)
    e_enc = safe_encode(le_event, event)
    p_enc = safe_encode(le_promo, promo)
    r_enc = safe_encode(le_rest, str(restaurant_id))
    cuis_enc = safe_encode(le_cuisine, meta.get("cuisine_type", "Unknown"))

    w_mult = WEATHER_MULT.get(weather, 1.0)
    e_mult = EVENT_MULT.get(event, 1.0)
    p_mult = PROMO_MULT.get(promo, 1.0)
    we_mult = WEEKEND_MULT.get(is_weekend_flag, 1.0)

    weather_stat = WEATHER_STATS.get(weather, {})
    rain_mm = float(weather_stat.get("rain_mm", DEFAULT_RAIN_MM))
    temp_c = float(weather_stat.get("temperature_c", DEFAULT_TEMP_C))

    df["restaurant_id"] = df["restaurant_id"].astype(str)
    recent = df[df["restaurant_id"] == str(restaurant_id)].sort_values(["date", "hour"])

    rows = []

    for h in OPERATING_HOURS:
        same_h = recent[recent["hour"] == h]["actual_covers"]

        lag1 = float(same_h.iloc[-1]) if len(same_h) >= 1 else 30.0
        lag24 = float(same_h.iloc[-2]) if len(same_h) >= 2 else lag1
        lag48 = float(same_h.iloc[-3]) if len(same_h) >= 3 else lag1
        lag168 = float(same_h.iloc[-8]) if len(same_h) >= 8 else lag1

        roll3 = float(recent["actual_covers"].iloc[-3:].mean()) if len(recent) >= 3 else lag1
        roll6 = float(recent["actual_covers"].iloc[-6:].mean()) if len(recent) >= 6 else lag1
        roll24 = float(recent["actual_covers"].iloc[-24:].mean()) if len(recent) >= 24 else lag1

        havg = HOUR_AVG.get((str(restaurant_id), h), 40.0)

        is_lunch = int(h in LUNCH_PEAK_HOURS)
        is_dinner = int(h in DINNER_PEAK_HOURS)
        is_off = int(h not in ACTIVE_SERVICE_HOURS)

        rows.append({
            "date": forecast_ts,
            "rest_enc": r_enc,
            "seating_capacity": meta["seating_capacity"],
            "avg_ticket_size": meta["avg_ticket_size"],
            "cuisine_enc": cuis_enc,
            "hour": h,
            "hour_sin": np.sin(2 * np.pi * h / 24),
            "hour_cos": np.cos(2 * np.pi * h / 24),
            "day_of_week": day_of_week,
            "month_sin": np.sin(2 * np.pi * month / 12),
            "month_cos": np.cos(2 * np.pi * month / 12),
            "week_of_year": week_of_year,
            "day_of_month": day_of_month,
            "is_lunch_peak": is_lunch,
            "is_dinner_peak": is_dinner,
            "is_off_peak": is_off,
            "is_weekend": is_weekend_flag,
            "is_weekend_mult": we_mult,
            "weather_enc": w_enc,
            "weather_mult": w_mult,
            "event_enc": e_enc,
            "event_mult": e_mult,
            "promo_enc": p_enc,
            "promo_mult": p_mult,
            "rain_mm": rain_mm,
            "temperature_c": temp_c,
            "peak_weather": (is_lunch + is_dinner) * w_mult,
            "peak_event": (is_lunch + is_dinner) * e_mult,
            "hour_avg_rest": havg,
            "lag_1h": lag1,
            "lag_24h": lag24,
            "lag_48h": lag48,
            "lag_168h": lag168,
            "roll_3h": roll3,
            "roll_6h": roll6,
            "roll_24h": roll24,
            "correction_ratio": 1.0,
            "reason_enc": 0,
        })

    return pd.DataFrame(rows)


def forecast_staff(covers_arr, restaurant_id, is_we, forecast_date):
    meta = get_rest_meta(restaurant_id)
    cap = meta["seating_capacity"]
    hours = OPERATING_HOURS
    forecast_ts = pd.to_datetime(forecast_date, errors="coerce")
    if pd.isna(forecast_ts):
        forecast_ts = hourly["date"].dropna().max() + pd.Timedelta(days=1)
    day_of_week = int(pd.Timestamp(forecast_ts).dayofweek)
    recs = []

    for i, h in enumerate(hours):
        c = float(covers_arr[i])
        h_sin = np.sin(2 * np.pi * h / 24)
        h_cos = np.cos(2 * np.pi * h / 24)
        is_lunch = int(h in LUNCH_PEAK_HOURS)
        is_dinner = int(h in DINNER_PEAK_HOURS)

        for (role, station), m in staff_models.items():
            feat = pd.DataFrame(
                [[h, h_sin, h_cos, day_of_week, is_we, is_lunch, is_dinner, c, cap]],
                columns=STAFF_FEATURES,
            )

            n = max(1, int(round(float(
                m.predict(staff_scalers[(role, station)].transform(feat))[0]
            ))))

            recs.append({
                "hour": h,
                "role": role,
                "station": station,
                "staff": n,
            })

    return pd.DataFrame(recs)


def forecast_inventory(total_covers, is_we):
    rows = []

    for ing, m in ing_models.items():
        meta = inventory[inventory["ingredient_name"] == ing].iloc[0]

        shelf = int(meta["shelf_life_days"])
        lead = int(meta["supplier_lead_time_days"])
        safety = 0.10 if shelf <= 3 else 0.20 if shelf <= 7 else 0.30
        usable = min(shelf, 7)
        urgency = lead / max(1, shelf)
        proj = total_covers * 0.05

        feat = pd.DataFrame(
            [[total_covers, shelf, lead, is_we, safety, usable, urgency, proj]],
            columns=INV_FEATURES,
        )

        qty = max(0.0, round(float(
            m.predict(ing_scalers[ing].transform(feat))[0]
        ), 2))

        priority = "HIGH" if shelf <= 3 else "MED" if shelf <= 7 else "LOW"

        rows.append({
            "ingredient": ing,
            "order_qty": qty,
            "shelf_life": shelf,
            "lead_time": lead,
            "safety_stock": f"{int(safety * 100)}%",
            "stockout_risk": round(float(meta["stockout_risk"]), 3),
            "priority": priority,
        })

    return pd.DataFrame(rows).sort_values("stockout_risk", ascending=False)


def plot_forecast(covers, staff_df):
    hours = OPERATING_HOURS
    covers = np.asarray(covers, dtype=float)

    has_conv = len(live_feedback_log) >= 2
    ncols = 3 if has_conv else 2

    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 5))
    axes = np.atleast_1d(axes).ravel()
    fig.suptitle(
        "Restaurant Resource Planning - Forecast Dashboard",
        fontsize=13,
        fontweight="bold",
    )

    ax = axes[0]
    colours = ["#E63946" if c == covers.max() else "#457B9D" for c in covers]

    ax.bar(hours, covers, color=colours, zorder=3)
    ax.axvspan(11.5, 14.5, alpha=0.12, color="orange", label="Lunch peak zone")
    ax.axvspan(18.5, 21.5, alpha=0.12, color="purple", label="Dinner peak zone")
    ax.set_title("Predicted Covers by Hour")
    ax.set_xlabel("Hour")
    ax.set_ylabel("Covers")
    ax.set_xticks(hours)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    ax2 = axes[1]
    if staff_df.empty:
        pivot = pd.DataFrame(index=hours)
    else:
        pivot = staff_df.groupby(["hour", "role"])["staff"].max().unstack(fill_value=0)
        pivot = pivot.reindex(hours, fill_value=0)

    if pivot.shape[1] > 0:
        pivot.plot(kind="bar", stacked=True, ax=ax2, colormap="Set2", width=0.8)
    else:
        ax2.text(0.5, 0.5, "No staff plan available", ha="center", va="center")
    ax2.set_title("Hourly Staff Plan by Role")
    ax2.set_xlabel("Hour")
    ax2.set_ylabel("Headcount")
    ax2.tick_params(axis="x", rotation=45)
    if pivot.shape[1] > 0:
        ax2.legend(loc="upper left", fontsize=8)

    if has_conv:
        ax3 = axes[2]

        rounds = [c["round"] for c in live_feedback_log[-20:]]
        preds = [c["predicted"] for c in live_feedback_log[-20:]]
        actuals = [c["actual"] for c in live_feedback_log[-20:]]
        errors = [c["total_error"] for c in live_feedback_log[-20:]]
        next_estimates = [c["next_estimate"] for c in live_feedback_log[-20:]]

        ax3.plot(rounds, preds, "b-o", label="Predicted", markersize=5)
        ax3.plot(rounds, next_estimates, "m-.D", label="Next calibrated", markersize=4)
        ax3.plot(rounds, actuals, "r--s", label="Actual", markersize=5)
        ax3.set_title("Live Convergence - Predicted vs Actual")
        ax3.set_xlabel("Your Feedback Round")
        ax3.set_ylabel("Total Covers")
        ax3.legend(fontsize=8)
        ax3.grid(alpha=0.3)

        ax3b = ax3.twinx()
        ax3b.plot(rounds, errors, "g:^", label="Total Error", markersize=4, alpha=0.7)
        ax3b.set_ylabel("Total Error", color="green")
        ax3b.tick_params(axis="y", labelcolor="green")
        ax3b.legend(loc="upper right", fontsize=7)

    plt.tight_layout()
    return fig


def figure_to_image(fig):
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())
    image = np.array(rgba)[..., :3]
    plt.close(fig)
    return image


# SECTION 10 - GRADIO UI
RESTAURANT_IDS = sorted(restaurants["restaurant_id"].astype(str).unique().tolist())
WEATHER_OPTIONS = sorted(df["weather_type"].dropna().astype(str).unique().tolist())
EVENT_OPTIONS = sorted(df["event_type"].dropna().astype(str).unique().tolist())
PROMO_OPTIONS = sorted(df["promo_type"].dropna().astype(str).unique().tolist())
REASON_OPTIONS = [
    str(v) for v in sorted(corr_feat["correction_reason"].dropna().astype(str).unique())
]
if not REASON_OPTIONS:
    REASON_OPTIONS = [DEFAULT_REASON]
DEFAULT_WEATHER_OPTION = DEFAULT_WEATHER if DEFAULT_WEATHER in WEATHER_OPTIONS else WEATHER_OPTIONS[0]
DEFAULT_EVENT_OPTION = DEFAULT_EVENT if DEFAULT_EVENT in EVENT_OPTIONS else EVENT_OPTIONS[0]
DEFAULT_PROMO_OPTION = DEFAULT_PROMO if DEFAULT_PROMO in PROMO_OPTIONS else PROMO_OPTIONS[0]
DEFAULT_REASON_OPTION = (
    DEFAULT_REASON if DEFAULT_REASON in REASON_OPTIONS else REASON_OPTIONS[0]
)


def run_forecast(
    restaurant_id,
    weather,
    event,
    promo,
    forecast_date,
    actual_covers,
    correction_reason,
):
    forecast_ts = pd.to_datetime(forecast_date, errors="coerce")
    if pd.isna(forecast_ts):
        forecast_ts = pd.to_datetime(DEFAULT_FORECAST_DATE)
    forecast_date = pd.Timestamp(forecast_ts).strftime("%Y-%m-%d")
    is_we = int(pd.Timestamp(forecast_ts).dayofweek >= 5)

    future_df = build_forecast_rows(restaurant_id, weather, event, promo, forecast_date, is_we)
    raw_covers = model.predict(scaler.transform(future_df[FEATURES])).clip(min=0)
    calibration_factor = get_calibration_factor(
        restaurant_id, weather, event, promo, is_we
    )
    covers = (raw_covers * calibration_factor).clip(min=0)

    total = int(covers.sum())
    raw_total = int(raw_covers.sum())
    peak_hour = int(np.argmax(covers)) + 8
    peak_val = int(covers.max())

    staff_df = forecast_staff(covers, restaurant_id, is_we, forecast_date)
    role_peak = staff_df.groupby("role")["staff"].max().to_dict()
    sta_peak = staff_df.groupby("station")["staff"].max().to_dict()

    inv_df = forecast_inventory(float(total), is_we)

    feedback_msg = "-  (enter actual covers above to trigger retraining)"

    if actual_covers and actual_covers > 0:
        feedback_msg = apply_feedback(
            str(restaurant_id),
            float(actual_covers),
            float(total),
            correction_reason,
            is_live=True,
            weather=weather,
            event=event,
            promo=promo,
            is_weekend_flag=is_we,
            forecast_rows=future_df,
            forecast_covers=covers,
        )

    meta = get_rest_meta(restaurant_id)
    est_revenue = round(total * meta["avg_ticket_size"], 0)

    role_rows = "".join(
        f"| {r} | **{n}** |\n"
        for r, n in role_peak.items()
    )

    station_rows = "".join(
        f"| {s} | **{n}** |\n"
        for s, n in sta_peak.items()
    )

    inventory_rows = "".join(
        f"| {r['ingredient']} | {r['order_qty']} | {r['shelf_life']}d | "
        f"{r['lead_time']}d | {r['safety_stock']} | {r['stockout_risk']} | "
        f"{r['priority']} |\n"
        for _, r in inv_df.iterrows()
    )

    conv_summary = ""

    if len(live_feedback_log) >= 1:
        last5 = live_feedback_log[-5:]
        conv_rows = "".join(
            f"| {c['round']} | {c['predicted']} | {c['actual']} | "
            f"{c['total_error']} | {c['total_error_pct']}% | "
            f"{c['calibration_factor']} | {c['next_estimate']} |\n"
            for c in last5
        )

        conv_summary = (
            "\n\n### Live Convergence (your last 5 inputs)\n"
            "| Round | Predicted | Actual | Total Error | Error % | Cal Factor | Next Estimate |\n"
            "|-------|-----------|--------|-------------|---------|------------|---------------|\n"
            f"{conv_rows}"
        )

    report = f"""
## Forecast - {restaurant_id} | {forecast_date} | {weather} | {event} | {promo}

### Demand Prediction
| Metric | Value |
|--------|-------|
| Total Covers | **{total}** |
| Raw Model Covers | **{raw_total}** |
| Live Calibration Factor | **{calibration_factor:.3f}** |
| Weekend Flag | **{is_we}** |
| Peak Hour | **{peak_hour}:00** ({peak_val} covers) |
| Estimated Revenue | **INR {est_revenue:,.0f}** |

### Staff Plan - by Role (peak headcount)
| Role | Staff Required |
|------|---------------|
{role_rows}

### Staff Plan - by Station (peak headcount)
| Station | Staff Required |
|---------|---------------|
{station_rows}

### Ingredient Orders (shelf-life aware)
| Ingredient | Order Qty | Shelf Life | Lead Time | Safety | Stockout Risk | Priority |
|-----------|-----------|-----------|-----------|--------|--------------|---------|
{inventory_rows}

### Feedback / Retraining
{feedback_msg}
{conv_summary}
"""

    fig = plot_forecast(covers, staff_df)
    chart_image = figure_to_image(fig)
    return report, chart_image


with gr.Blocks(title="Restaurant Resource Planning System v2") as demo:
    gr.Markdown("""
# Restaurant Resource Planning System v2
### Self-Learning Forecaster - Covers, Staff by Role and Station, Inventory with Shelf-Life
""")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("#### Forecast Inputs")

            rest_dd = gr.Dropdown(RESTAURANT_IDS, value=DEFAULT_RESTAURANT, label="Restaurant")
            forecast_date_in = gr.Textbox(
                value=DEFAULT_FORECAST_DATE,
                label="Forecast Date (YYYY-MM-DD)",
            )
            weather_dd = gr.Dropdown(WEATHER_OPTIONS, value=DEFAULT_WEATHER_OPTION, label="Weather")
            event_dd = gr.Dropdown(EVENT_OPTIONS, value=DEFAULT_EVENT_OPTION, label="Event Type")
            promo_dd = gr.Dropdown(PROMO_OPTIONS, value=DEFAULT_PROMO_OPTION, label="Promo Type")

            gr.Markdown("#### Manager Feedback")
            gr.Markdown(
                "_Enter actual covers to retrain. "
                "Must be **50%-200%** of predicted total to be accepted._"
            )

            actual_in = gr.Number(
                value=0,
                label="Actual Covers Yesterday (0 = skip)",
            )

            reason_dd = gr.Dropdown(
                REASON_OPTIONS,
                value=DEFAULT_REASON_OPTION,
                label="Correction Reason",
            )

            run_btn = gr.Button("Run Forecast", variant="primary")

        with gr.Column(scale=2):
            out_text = gr.Markdown()
            out_plot = gr.Image(type="numpy", label="Forecast Charts")

    run_btn.click(
        fn=run_forecast,
        inputs=[
            rest_dd,
            weather_dd,
            event_dd,
            promo_dd,
            forecast_date_in,
            actual_in,
            reason_dd,
        ],
        outputs=[out_text, out_plot],
    )

    gr.Markdown("""
---
**Convergence Test:** Enter the same actual value 4 times in a row.
The predicted total should move toward your actual, and the convergence
chart (3rd panel) will appear after your 2nd feedback round.
""")

print("\nLaunching...")
demo.launch(debug=True, share=True)


Imports complete
Data loaded - hourly:(13500, 13) staff:(54000, 9) inv:(9000, 10) corrections:(647, 6)

Data-driven constants:
  Weather multipliers : {'Clear': 1.0, 'Cloudy': 0.9949412337292505, 'Rain': 0.792023073558556, 'Storm': 0.6400741610556822}
  Event multipliers   : {'Festival': 1.0, 'LocalEvent': 1.0019485045800438, 'SportsMatch': 0.9758330566454524}
  Promo multipliers   : {'FestivalOffer': 1.0, 'HappyHour': 0.9996404570434908, 'WeekendDeal': 1.0332922999670826}
  Weekend multiplier  : {0: 1.0, 1: 1.1656243404097293}
  Operating hours     : [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
  Lunch peak hours    : [12, 13, 14]
  Dinner peak hours   : [19, 20, 21]

  FEATURE ENGINEERING
Features engineered: 37

  DEMAND FORECASTING MODEL - XGBoost v2
  Best iteration : 191
  Train MAE      : 4.5
  Test  MAE      : 5.61 - held-out
  Test  RMSE     : 8.04

  Top 10 features:
    peak_weather               0.2473
    hour_avg_rest              0.1751
    avg_ticket_size 